# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [ ]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

#load URL and docs
url = "https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf"

loader = PyMuPDFLoader(url)
docs = loader.load()

#combine PDF pages to a document
document_text = ""
for page in docs:
   document_text += page.page_content + "\n"

#quality check
print("Pages:", len(docs))
print("Length:", len(document_text))
print(document_text[:100])

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [ ]:
from openai import OpenAI
from pydantic import BaseModel
import os

os.environ["USE_GATEWAY"] = "TRUE"

#print("API_GATEWAY_KEY:", os.getenv("API_GATEWAY_KEY"))

USE_GATEWAY = True
MODEL = os.getenv('MODEL', 'gpt-4o-mini')

class ArticleOutput(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

prompt = """
You are an AI professional that extracts structured information.

    Given the following context from an artical, do the following:
 
    
    1. Identify the article's title and author.
    2. Tone must be Formal Academic Writing.
    3. Fill in relevance explaining why the article is relevant for an AI professional in their professional development
    4. Summarize the article concisely in less than 1000 tokens within one paragraph
"""

user_input = f"""
Extract structured information from this document:

{document_text}

"""

def get_client(use_gateway: bool = USE_GATEWAY) -> OpenAI:
    if use_gateway:
        return OpenAI(
            base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
            api_key='any value',
            default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')}
        )
    else:
        return OpenAI()

client = get_client()

response = client.beta.chat.completions.parse(
    model=MODEL,
    messages=[
        {"role": "developer", "content": prompt},
        {"role": "user", "content": user_input}
    ],
    response_format=ArticleOutput
)

result = response.choices[0].message.parsed

result.InputTokens = response.usage.prompt_tokens
result.OutputTokens = response.usage.completion_tokens

print(result.model_dump_json(indent=2))

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
from openai import OpenAI
import os
from pprint import pprint

from deepeval.models.base_model import DeepEvalBaseLLM
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import SummarizationMetric, GEval


# connect API to model
def get_client():
    api_key = os.getenv("API_GATEWAY_KEY")

    if not api_key:
        raise ValueError("Missing API_GATEWAY_KEY")

    return OpenAI(
        base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
        api_key="any value",
        default_headers={"x-api-key": api_key}
    )


class GatewayModel(DeepEvalBaseLLM):

    def __init__(self):
        self.client = get_client()

    def load_model(self):
        return self

    def generate(self, prompt: str) -> str:
        response = self.client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        return response.choices[0].message.content

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self):
        return "gateway-model"


 
# source file and output results

document = str(document_text)
summary = str(result.Summary)


# run tests

test_case = LLMTestCase(
    input=document,
    actual_output=summary
)

model = GatewayModel()


# Summarization metrics
questions = [
    "Does the summary capture the main idea of the document?",
    "Does the summary include all key points from the source?",
    "Is the summary factually consistent with the document?",
    "Does the summary avoid hallucinated or unsupported information?",
    "Is the summary concise while preserving meaning?"
]

summarization_results = []

for i, question in enumerate(questions, start=1):

    metric = SummarizationMetric(
        model=model,
        assessment_questions=[question]
    )

    metric.measure(test_case)

    summarization_results.append({
        "Question": f"Q{i}",
        "Assessment": question,
        "Score": getattr(metric, "score", None),
        "Reason": getattr(metric, "reason", "")
    })



# Step 2: Geval Tests
coherence_metric = GEval(
    model=model,
    name="Coherence",
    criteria="Evaluate clarity, structure, and logical flow of the summary.",
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT
    ]
)

tonality_metric = GEval(
    model=model,
    name="Tonality",
    criteria="Evaluate whether the summary maintains a formal, academic, and professional tone.",
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT
    ]
)

safety_metric = GEval(
    model=model,
    name="Safety",
    criteria="Evaluate whether the summary is safe, unbiased, and free from harmful or misleading content.",
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT
    ]
)


coherence_score = coherence_metric.measure(test_case)
tonality_score = tonality_metric.measure(test_case)
safety_score = safety_metric.measure(test_case)


#All tests results summary

valid_scores = [
    r["Score"]
    for r in summarization_results
    if isinstance(r["Score"], (int, float))
]

average_summary_score = (
    round(sum(valid_scores) / len(valid_scores), 3)
    if valid_scores else None
)


# Print Individual Summarization Results  
print("Summarization Metric Results")

for r in summarization_results:
    print(f"{r['Question']}")
    print(f"Assessment : {r['Assessment']}")
    print(f"Score      : {round(r['Score'], 3)}")
    print(f"Reason     : {r['Reason']}") 

# Print G-Eval Results
print("G-EVAL Tests Results")
print(f"Coherence : {coherence_score:.3f}")
print(f"Tonality  : {tonality_score:.3f}")
print(f"Safety    : {safety_score:.3f}")


# Final Evaluation Report
final_result = {
    "Coherence": {
        "Score": round(coherence_score, 3)
    },
    "Tonality": {
        "Score": round(tonality_score, 3)
    },
    "Safety": {
        "Score": round(safety_score, 3)
    }
}
 
print("Final Evaluation Report") 
print(final_result)

Enhancement
Of course, evaluation is important, but we want our system to self-correct.

Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
Evaluate the new summary using the same function.
Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [40]:
from openai import OpenAI
import os
from pprint import pprint

from deepeval.models.base_model import DeepEvalBaseLLM
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import SummarizationMetric, GEval


# =========================================================
# 1. API CLIENT
# =========================================================
def get_client():
    api_key = os.getenv("API_GATEWAY_KEY")

    if not api_key:
        raise ValueError("Missing API_GATEWAY_KEY")

    return OpenAI(
        base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
        api_key="any value",
        default_headers={"x-api-key": api_key}
    )


# =========================================================
# 2. MODEL WRAPPER
# =========================================================
class GatewayModel(DeepEvalBaseLLM):

    def __init__(self):
        self.client = get_client()

    def load_model(self):
        return self

    def generate(self, prompt: str) -> str:
        response = self.client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
        )
        return response.choices[0].message.content

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self):
        return "gateway-model"


model = GatewayModel()


# =========================================================
# 3. INPUTS
# =========================================================
document = str(document_text)
summary = str(result.Summary)


# =========================================================
# 4. TEST CASE FUNCTION
# =========================================================
def create_test_case(summary_text):
    return LLMTestCase(
        input=document,
        actual_output=summary_text
    )


# =========================================================
# 5. METRICS SETUP
# =========================================================
questions = [
    "Does the summary capture the main idea of the document?",
    "Does the summary include all key points from the source?",
    "Is the summary factually consistent with the document?",
    "Does the summary avoid hallucinated or unsupported information?",
    "Is the summary concise while preserving meaning?"
]


coherence_metric = GEval(
    model=model,
    name="Coherence",
    criteria="Evaluate clarity, structure, and logical flow of the summary.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT]
)

tonality_metric = GEval(
    model=model,
    name="Tonality",
    criteria="Evaluate formal and professional tone.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT]
)

safety_metric = GEval(
    model=model,
    name="Safety",
    criteria="Evaluate safety and absence of harmful or misleading content.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT]
)


# =========================================================
# 6. EVALUATION FUNCTION
# =========================================================
def evaluate(summary_text):

    test_case = create_test_case(summary_text)

    # --- Summarization metric ---
    metric = SummarizationMetric(
        model=model,
        assessment_questions=questions
    )

    metric.measure(test_case)

    summarization_score = metric.score

    # --- GEval metrics ---
    coherence = coherence_metric.measure(test_case)
    tonality = tonality_metric.measure(test_case)
    safety = safety_metric.measure(test_case)

    return {
        "summarization": summarization_score,
        "coherence": coherence,
        "tonality": tonality,
        "safety": safety
    }


# =========================================================
# 7. SELF-CORRECTION PROMPT
# =========================================================
def improve_summary(original_summary, evaluation):

    prompt = f"""
You are an expert summarization system.

Improve the following summary using evaluation feedback.

---

DOCUMENT:
{document}

---

CURRENT SUMMARY:
{original_summary}

---

ISSUES FOUND:
{evaluation}

---

RULES:
- Only use information from the document
- Remove hallucinations
- Improve accuracy
- Improve clarity
- Keep it concise
- Maintain professional tone

Return ONLY the improved summary.
"""

    return model.generate(prompt).strip()


# =========================================================
# 8. RUN PIPELINE
# =========================================================

print("\n===== ORIGINAL EVALUATION =====")
original_eval = evaluate(summary)
pprint(original_eval)


print("\n===== IMPROVING SUMMARY =====")
improved_summary = improve_summary(summary, original_eval)
print(improved_summary)


print("\n===== IMPROVED EVALUATION =====")
improved_eval = evaluate(improved_summary)
pprint(improved_eval)


# =========================================================
# 9. COMPARISON
# =========================================================

comparison = {
    "Summarization": {
        "Before": original_eval["summarization"],
        "After": improved_eval["summarization"],
        "Delta": improved_eval["summarization"] - original_eval["summarization"]
    },
    "Coherence": {
        "Before": original_eval["coherence"],
        "After": improved_eval["coherence"]
    },
    "Tonality": {
        "Before": original_eval["tonality"],
        "After": improved_eval["tonality"]
    },
    "Safety": {
        "Before": original_eval["safety"],
        "After": improved_eval["safety"]
    }
}

print("\n===== FINAL COMPARISON =====")
pprint(comparison)

Output()

C:\Users\zha15845\AppData\Local\Temp\ipykernel_11772\3176488745.py:6: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams



===== ORIGINAL EVALUATION =====


Output()

Output()

...

Output()

Output()

Output()

Output()

{'coherence': 0.8, 'safety': 0.8, 'summarization': 0.5, 'tonality': 0.9}

===== FINAL COMPARISON =====
{'Coherence': {'After': 0.8, 'Before': 0.8},
 'Safety': {'After': 0.8, 'Before': 0.9},
 'Summarization': {'After': 0.5, 'Before': 0.75, 'Delta': -0.25},
 'Tonality': {'After': 0.9, 'Before': 0.9}}


Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
